In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # Paid alternative
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate

In [ ]:
# Locate the project from either the notebook folder or the workspace root.
cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd / "Week2-Session1",
    cwd / "Mastering-Agentic-AI-Week2-Session1" / "Week2-Session1",
]
project_dir = next(
    (directory for directory in candidates
     if (directory / "shopeasy_knowledge_base.json").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError(
        "Cannot find shopeasy_knowledge_base.json. Start the kernel from the "
        "Master_AgenticAI workspace or the Week2-Session1 folder."
    )
load_dotenv(project_dir / ".env")


![Diagram](./images/00_Basic_RAG_Pipeline.png)

# Data Extraction

The first step in any RAG pipeline is to get the data you want to work with. Here we're building a customer-support assistant for **ShopEasy**, a multi-category e-commerce platform.

Our knowledge base lives in `shopeasy_knowledge_base.json` — 51 internal support documents across five types (`runbook`, `past_ticket`, `product_doc`, `bug_report`, `faq`) spanning five product areas (`payments`, `returns`, `shipping`, `orders`, `account`).

Each entry has a `content` string and a `metadata` object (`doc_type`, `product_area`, `priority`, `platform`, `customer_tier`, `status`, `title`). We load the JSON and wrap each entry in a LangChain `Document`, carrying the metadata along so it's available at retrieval time.

![Data Extraction](./images/01_data_extraction.png)

In [ ]:
# Load the ShopEasy knowledge base from JSON and wrap each entry in a Document.
with open(project_dir / "shopeasy_knowledge_base.json", encoding="utf-8") as f:
    kb = json.load(f)

docs = [
    Document(page_content=entry["content"], metadata=entry["metadata"])
    for entry in kb
]

In [ ]:
len(docs) # Number of documents loaded from the ShopEasy knowledge base

In [ ]:
# Let's inspect the metadata of one of the loaded documents.
# Each support doc carries structured metadata (doc_type, product_area, priority,
# platform, customer_tier, status, title) that we'll later use for filtering.

docs[0].page_content
# docs[0].metadata

In [ ]:
# To get a better sense of the content we've loaded,
# this code block defines a helper function wrap_text to format the text and then prints the content of each document.

def wrap_text(text, width=80):
    return '\n'.join([text[i:i+width] for i in range(0, len(text), width)])

for doc in docs:
    print(wrap_text(doc.page_content))
    print("-"*100)

# Chunk the data

Now that we have our documents, the next step is to split them into smaller chunks. This is important for a few reasons:

__Vector search efficiency__: Smaller chunks are easier to search and retrieve.

__Context window limitations__: LLMs have a limited context window, so we need to make sure the retrieved information fits within that window.

__Relevance__: Smaller chunks are more likely to be focused on a specific topic, which improves the relevance of the retrieved information.

We'll use the `RecursiveCharacterTextSplitter` to split our documents. This splitter tries to split text on a series of characters (like newlines, spaces, etc.) in a recursive manner.

- `chunk_size=500`: This sets the maximum size of each chunk to 500 characters. Our support docs are fairly short, so a smaller chunk size means the longer runbooks and tickets actually split into several focused chunks instead of staying as one big blob.

- `chunk_overlap=100`: This creates an overlap of 100 characters between consecutive chunks. This helps to ensure that we don't lose any important context at the boundaries of our chunks.

- `add_start_index=True`: This will add the starting index of the chunk in the original document to the metadata.

Importantly, the metadata from each parent document (doc_type, product_area, title, …) is copied onto **every** chunk it produces, so we never lose track of where a chunk came from.

![Chunk the Data](./images/02_chunking.png)

In [ ]:
# chunk the data

# ref: https://docs.langchain.com/oss/python/integrations/splitters
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # chunk size (characters)
    chunk_overlap=100,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)

splits = text_splitter.split_documents(docs)

In [ ]:
# See what chunking did: 51 documents become a larger number of smaller chunks.
print(f"Documents in:  {len(docs)}")
print(f"Chunks out:    {len(splits)}")
print("-" * 100)

# Find the longest document and show how it got split (with overlap + start_index).
longest_idx = max(range(len(docs)), key=lambda i: len(docs[i].page_content))
parent = docs[longest_idx]
print(f'Longest doc: "{parent.metadata["title"]}" ({len(parent.page_content)} chars)\n')

child_chunks = [
    s for s in splits if s.metadata.get("title") == parent.metadata["title"]
]
print(f"It split into {len(child_chunks)} chunks:\n")
for i, chunk in enumerate(child_chunks):
    start = chunk.metadata["start_index"]
    print(f"[chunk {i} | start_index={start} | {len(chunk.page_content)} chars]")
    print(chunk.page_content)
    print("-" * 100)

In [ ]:
# for split in splits:
#     print(wrap_text(split.page_content))
#     print("-"*100)

In [ ]:
# splits[2].page_content
splits[2].metadata

### Index with local embeddings
Ollama runs `nomic-embed-text` locally (768 dimensions by default).
Create a separate dense cosine Pinecone index and set `PINECONE_LOCAL_INDEX_NAME`.
The notebook checks the actual embedding dimension before writing vectors.
Local vectors use namespace `shopeasy-local-rag`; do not mix them with OpenAI vectors.
See `LOCAL-MODELS.txt` for installation and model download commands.


![Chunk the Data](./images/03_indexing.png)

<!-- ![Indexing](./images/03_indexing.png) -->

In [ ]:
# Local inference requires no OpenAI key. Pinecone remains a hosted service.
required_variables = ("PINECONE_API_KEY", "PINECONE_LOCAL_INDEX_NAME")
missing = [name for name in required_variables if not os.getenv(name, "").strip()]
if missing:
    raise ValueError("Missing configuration: " + ", ".join(missing)
                     + ". Set these values in Week2-Session1/.env.")

ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)
embeddings = OllamaEmbeddings(
    model=os.getenv("OLLAMA_EMBED_MODEL", "nomic-embed-text"),
    base_url=ollama_base_url,
)
try:
    embedding_dimension = len(embeddings.embed_query("Embedding dimension check"))
except Exception as exc:
    raise RuntimeError(
        "Local embeddings failed. Start Ollama and run: ollama pull "
        + os.getenv("OLLAMA_EMBED_MODEL", "nomic-embed-text")
    ) from exc

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = os.environ["PINECONE_LOCAL_INDEX_NAME"]
description = pc.describe_index(index_name)
if description.dimension != embedding_dimension:
    raise ValueError(
        f"Index {index_name!r} has {description.dimension} dimensions; "
        f"the local embedding model needs {embedding_dimension}. "
        "Create a separate dense cosine index with that dimension."
    )
index = pc.Index(index_name)
vector_store = PineconeVectorStore(
    index=index, embedding=embeddings, namespace="shopeasy-local-rag",
)


In [ ]:
# Index the chunks before retrieval. Embeddings are generated locally by Ollama.
# Stable IDs overwrite the same chunks when this cell is rerun.
from hashlib import sha256

chunk_ids = [
    sha256(json.dumps(
        {"content": chunk.page_content, "metadata": chunk.metadata},
        sort_keys=True, ensure_ascii=False,
    ).encode("utf-8")).hexdigest()
    for chunk in splits
]
document_ids = vector_store.add_documents(documents=splits, ids=chunk_ids)
print(f"Indexed {len(document_ids)} chunks in shopeasy-local-rag.")
# Pinecone is eventually consistent; wait briefly before running retrieval.


In [ ]:
# Let's retrieve a record from the Pinecone index by its ID to confirm it was indexed.
# Note: Pinecone is eventually consistent, so a just-added vector may take a
# moment to become fetchable.
# index.fetch(ids=[document_ids[10]], namespace="shopeasy-local-rag")

# Retrieval

With our documents indexed, we can now perform retrieval. The goal of this step is to find the most relevant document chunks for a given customer issue.

First, we'll set up our LLM and a prompt template.

- `ChatOllama` runs `qwen3.5:2b` locally for generation. No OpenAI key is required.

- `template = ...`: This is the prompt template that combines the retrieved context with the customer's issue. The `{context}` and `{question}` placeholders are filled in at generation time.

![Retrieval](./images/04_retrieval.png)

In [ ]:
#configure the llm
# llm = ChatOpenAI(model="gpt-4.1-mini")  # Paid alternative
llm = ChatOllama(
    model=os.getenv("OLLAMA_CHAT_MODEL", "qwen3.5:2b"),
    base_url=ollama_base_url, temperature=0,
)

#set the prompt template
template = """You are a customer-support assistant for ShopEasy, an e-commerce platform.
Use the following pieces of retrieved internal knowledge-base context to help resolve the customer's issue.
If the context doesn't contain the answer, say you don't have that information rather than guessing.
Be concise and practical: state the likely cause and the next step the agent should take.

Context:
{context}

Customer issue: {question}

Support guidance:"""

rag_prompt_template = PromptTemplate.from_template(template)

In [ ]:
# A sample customer support ticket (README Query 1).
user_question = "Customer says their order never arrived even though tracking shows delivered"

In [ ]:
# Now, we'll use the vector store's similarity_search method to find the top 5 most similar chunks to the customer issue.
retrieved_docs = vector_store.similarity_search(user_question, k=5)

In [ ]:
# Review the retrieved chunks. We print the metadata header so we can SEE the noise:
# semantic search will surface relevant shipping docs, but may also pull in
# returns/refund docs because the language overlaps.
for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['doc_type']} / {m['product_area']}] {m['title']}")
    print(doc.page_content)
    print("-" * 100)

In [ ]:
# The similarity_search_with_score method returns the chunks along with their similarity scores.
vector_store.similarity_search_with_score(user_question, k=5)

# Generation

The final step is to use the retrieved documents to generate an answer to the customer's issue.

We'll combine the content of the retrieved documents and use our prompt template to create a final prompt for the LLM.


![Generation](./images/05_generation.png)

In [ ]:
#generate answer

docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
prompt = rag_prompt_template.invoke({"question": user_question, "context": docs_content})
response = llm.invoke(prompt)

In [ ]:
#generated response
response.content

In [ ]:
# We can also include citations in our response by extracting the source from the metadata of the retrieved docs.
# This dataset has no "source" URL, so we cite each doc by its title and doc_type instead.

sources = [f"{doc.metadata['title']} ({doc.metadata['doc_type']})" for doc in retrieved_docs]

print(f"Sources: {sources}\n\n")
print(f'Answer: {response.content}')

### Langchain Retreiver

LangChain provides a `Retriever` interface, which is a more general way to retrieve documents. A vector store can be used as the backbone of a retriever, but there are other types of retrievers as well.

Here, we're creating a retriever from our vector store. We can also specify search arguments like k (the number of documents to retrieve) and search_type.

In [ ]:
# Ref: https://python.langchain.com/docs/concepts/retrievers/
retriever = vector_store.as_retriever(search_kwargs={"k": 5}, search_type='similarity')

retrieved_docs = retriever.invoke(user_question)

for doc in retrieved_docs:
    m = doc.metadata
    print(f"[{m['doc_type']} / {m['product_area']}] {m['title']}")
    print(doc.page_content)
    print("-" * 100)

![Overall](./images/06_put_it_together.png)

In [ ]:
# Now, let's put it all together in a single function.
def generate_answer(user_question):
    #retrieve the relevant docs
    retriever = vector_store.as_retriever(search_kwargs={"k": 5}, search_type='similarity')
    retrieved_docs = retriever.invoke(user_question)

    #generate
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = rag_prompt_template.invoke({"question": user_question, "context": docs_content})
    response = llm.invoke(prompt)

    return response.content

generate_answer("Customer says their order never arrived even though tracking shows delivered")

### More sample tickets — watch the noise

Plain semantic search handles natural language well, but when a query spans multiple product areas it casts a wide net. Run these and inspect the `[doc_type / product_area]` headers to see shipping, payments, and returns docs all mixed together — exactly the problem that metadata filtering (Notebook 2) and hybrid search (Notebook 3) will fix.

In [ ]:
# Helper to put the retrieved chunks on screen before the answer, so we can SEE the noise.
# Each hit shows its rank, [doc_type / product_area], title, similarity score and a one-line
# preview, followed by a tally of which product areas the results came from.
from collections import Counter
from textwrap import shorten

def show_retrieved(user_question, k=5):
    results = vector_store.similarity_search_with_score(user_question, k=k)

    print(f'Query: "{user_question}"\n')
    for rank, (doc, score) in enumerate(results, start=1):
        m = doc.metadata
        print(f"{rank}. [{m['doc_type']} / {m['product_area']}] {m['title']}  (score: {score:.3f})")
        print(f"   {shorten(doc.page_content, width=110, placeholder=' …')}")

    areas = Counter(doc.metadata["product_area"] for doc, _ in results)
    print("\nProduct areas retrieved:", ", ".join(f"{area} ×{n}" for area, n in areas.most_common()))
    print("-" * 100)

In [ ]:
# Query 2: a broad query returns broadly relevant results — returns runbook, return
# policy, return FAQ, plus past return tickets... and maybe warranty/billing noise.
query = "Customer wants to return a product they bought last month"
show_retrieved(query)
print(f"Answer: {generate_answer(query)}")

In [ ]:
# Query 3: an issue that spans payments + shipping + returns at once — semantic
# search casts a wide net across all three areas, making it hard to focus.
query = "Customer is upset about being charged but never receiving their product"
show_retrieved(query)
print(f"Answer: {generate_answer(query)}")